# Gemma 4 Classroom Adaptation: One Science Lesson, Three Reading Levels, Zero Cloud Dependence

## Fine-tuning Gemma 4 with Unsloth to help middle school science teachers serve mixed reading levels in low-bandwidth classrooms

## Project Links

- Public code repository: [fakepixels/gemma4-for-education](https://github.com/fakepixels/gemma4-for-education)
- Training framework: **Unsloth LoRA** on `google/gemma-4-E4B-it`
- This notebook is the project writeup. The GitHub repo is the source of truth for training, evaluation, and demo code.

## Problem

In many classrooms, one teacher has to support students reading at very different levels from the same lesson. That problem is especially difficult in science, where simplifying too much can distort the content, while keeping the original wording can leave some students behind. It becomes even harder in low-connectivity settings, where cloud-first tools may be slow, unreliable, or unsuitable for the classroom workflow.

Our project focuses on one narrow, practical use case: a teacher pastes one middle school science lesson, and the system generates `below`, `on`, and `above` reading-level versions while preserving the same core science facts. The teacher remains fully in control and reviews every output.

## Why Gemma 4

Gemma 4 was a strong fit for this project because the challenge is not open-ended tutoring. It is controlled transformation. We needed a model that could be post-trained to follow a precise classroom format, preserve required facts, and support a local-first deployment story for low-bandwidth environments.

## Method

We fine-tuned `google/gemma-4-E4B-it` using **Unsloth LoRA**. The training task was single-target rewriting: each example contained a source lesson, a target level label (`below`, `on`, or `above`), a list of must-keep facts, and a reference adaptation. We standardized the output into two sections:

- `Adapted Lesson`
- `Key Concepts Preserved`

This made the task easier to control during training, evaluation, and demo inference. The adapted lesson is student-facing. The key concepts section is teacher-facing and acts as a quick factual check.

## Dataset

The dataset is a compact middle school science adaptation corpus built around topics such as ecosystems, cells, atoms and molecules, electric circuits, water cycle, weather, energy, and matter. Each source lesson was expanded into three reading-level versions. We split the data by source lesson rather than by rewrite variant, so the held-out set tested generalization to unseen passages rather than memorization of paraphrases.

## Reproducibility

The full project can be reproduced from the public repository. The writeup notebook does not contain the entire training stack inline; instead, it documents the benchmark and links to the repo that runs the pipeline end to end.

High-level reproduction flow:

1. Clone the public repo
2. Install the local or CUDA environment
3. Build the processed dataset from the raw JSON files
4. Run the first adaptive Unsloth training pass
5. Run base-vs-tuned evaluation and summarize the benchmark

In [ ]:
# Exact commands used for the main CUDA run
git clone https://github.com/fakepixels/gemma4-for-education.git
cd gemma4-for-education
cp .env.example .env
bash scripts/setup_cuda_env.sh
PYTHONPATH=src python scripts/prepare_dataset.py --source data/raw --output-dir data/processed
PYTHONPATH=src python scripts/optimize_first_run.py --mode run --goal benchmark
PYTHONPATH=src python scripts/evaluate.py --config configs/eval.first_run.yaml --output artifacts/evals/first_run_eval.json
PYTHONPATH=src python scripts/summarize_eval.py --input artifacts/evals/first_run_eval.json --output artifacts/evals/first_run_summary.md

In [ ]:
from pathlib import Path
import json

manifest_path = Path('../data/processed/dataset_manifest.json')
if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text())
    manifest
else:
    print(f'Missing dataset manifest: {manifest_path.resolve()}')

## Key Artifacts

The reproducible pipeline writes the following artifacts:

- `data/processed/dataset_manifest.json`
- `artifacts/tuning/optimization_result.yaml`
- `artifacts/adapters/first-run/`
- `artifacts/evals/first_run_eval.json`
- `artifacts/evals/first_run_summary.md`

These files are the basis for the benchmark numbers and qualitative examples reported in this writeup.

## Evaluation

We compared base Gemma 4 against the fine-tuned adapter on a held-out benchmark. The evaluation measured:

- **Fact coverage:** whether the adapted lesson preserved required science facts
- **Reading-level alignment:** whether the adapted lesson matched the requested level
- **Teacher usefulness:** a classroom-oriented proxy combining factual fidelity, level control, and output completeness

We also updated the scoring pipeline to better reflect the real product. The benchmark now scores the student-facing lesson separately from the teacher note, extracts the final assistant answer cleanly, and uses a science-aware rubric for level control so necessary domain vocabulary is not unfairly penalized.

## Benchmark Results

| Metric | Base Gemma 4 | Tuned Gemma 4 | Delta |
|---|---:|---:|---:|
| Avg fact coverage | 0.417 | 1.000 | +0.583 |
| Avg teacher usefulness | 0.408 | 0.967 | +0.559 |
| Within target band rate | 0.417 | 0.833 | +0.416 |

In [ ]:
eval_path = Path('../artifacts/evals/first_run_eval.json')
if eval_path.exists():
    report = json.loads(eval_path.read_text())
    {
        'base': report['base']['summary'],
        'tuned': report['tuned']['summary'],
    }
else:
    print(f'Missing eval artifact: {eval_path.resolve()}')

These gains are meaningful because the project is intentionally narrow. We are not benchmarking general educational QA or open-ended tutoring. We are measuring one transformation: can the model reliably turn a teacher's science lesson into a level-appropriate classroom version without changing the science?

## Qualitative Example

One of the clearest held-out examples was **`electric_circuits_001` at the `on` level**. Under the same prompt, the base model failed to produce a usable adapted lesson. The tuned model returned a complete structured response, preserved all five required circuit facts, and matched the requested classroom format.

We saw the same pattern on other held-out `on`-level examples, including **`cells_001`** and **`atoms_molecules_001`**. In each case, the tuned model was more reliable at producing a classroom-ready output while preserving the science content.

In [ ]:
if eval_path.exists():
    report = json.loads(eval_path.read_text())
    for base_row, tuned_row, base_ex, tuned_ex in zip(report['base']['rows'], report['tuned']['rows'], report['base_examples'], report['tuned_examples']):
        if base_row['source_id'] == 'electric_circuits_001' and base_row['target_level'] == 'on':
            print('Base summary:', base_row)
            print()
            print('Tuned summary:', tuned_row)
            print()
            print('Base raw output:
')
            print(base_ex['generated_text'])
            print('
' + '=' * 80 + '
')
            print('Tuned raw output:
')
            print(tuned_ex['generated_text'])
            break

## Demo

The demo is intentionally simple. A teacher pastes one science lesson and receives:

- a below-level adapted lesson
- an on-level adapted lesson
- an above-level adapted lesson
- a teacher note summarizing preserved concepts

This supports the story we want to tell in the submission: one lesson becomes three usable versions quickly, without relying on a cloud-heavy workflow.

## Limitations and Next Steps

Our strongest current gains are in reliability, structure-following, and factual preservation. The next improvement target is stronger stylistic separation at the level extremes, especially for some `below`-level examples where the base model still simplifies more aggressively. The next step is a focused data pass that sharpens low-level simplification and high-level enrichment without weakening factual control.

## Impact

The value of this project is not novelty for its own sake. It is practical leverage for a real classroom constraint. A low-bandwidth teacher workflow needs outputs that are reliable, grounded, and fast to review. Fine-tuning Gemma 4 let us move from a generic model to one that is better aligned with that exact need.